# FDCPA Rule Classifier — Three-Way Evaluation

Compare three models on the held-out test set:
1. **o3-mini** — ceiling baseline (OpenAI API)
2. **Qwen2.5-3B-Instruct (base)** — zero-shot baseline (local)
3. **Qwen2.5-3B-Instruct + QLoRA adapter** — fine-tuned model (local)

**Prerequisites:**
- GPU with 16GB+ VRAM
- `OPENAI_API_KEY` set in environment
- Trained adapter at `./checkpoints/final`

In [ ]:
# Setup: install deps and add module path
!pip install -q transformers peft trl bitsandbytes accelerate openai scikit-learn matplotlib seaborn tqdm python-dotenv datasets

import sys, os
from pathlib import Path

# Try local path first, then Kaggle
if Path('../src/fdcpa_classifier').is_dir():
    sys.path.insert(0, '..')
elif Path('/kaggle/working/fdcpa-rule-classifier/src').is_dir():
    sys.path.insert(0, '/kaggle/working/fdcpa-rule-classifier/src')
else:
    raise ImportError('Cannot find fdcpa_classifier module. Clone the repo or add src/ to sys.path.')

from dotenv import load_dotenv
load_dotenv()

## 1. Load Test Set

In [ ]:
import pandas as pd
from fdcpa_classifier import load_jsonl

test_examples = load_jsonl('../data/test.jsonl')
df_test = pd.DataFrame(test_examples)

print(f"Test set: {len(test_examples)} examples")
print(f"\nRule distribution:")
print(df_test['rule_id'].value_counts().sort_index())
print(f"\nVerdict balance:")
print(df_test['verdict'].value_counts())

## 2. Run Fine-tuned Model Inference

In [ ]:
from fdcpa_classifier.eval import run_inference_finetuned

ft_results = run_inference_finetuned(
    model_path='../checkpoints/final',
    base_model='Qwen/Qwen2.5-3B-Instruct',
)
print(f"Fine-tuned: {len(ft_results)} predictions")

## 3. Run Base Model Inference

In [ ]:
from fdcpa_classifier.eval import run_inference_base

base_results = run_inference_base(
    model_name='Qwen/Qwen2.5-3B-Instruct',
)
print(f"Base model: {len(base_results)} predictions")

## 4. Run OpenAI Baseline Inference

In [ ]:
from fdcpa_classifier.eval import run_inference_openai

openai_results = run_inference_openai()
print(f"OpenAI: {len(openai_results)} predictions")

## 5. Comparison Table

In [ ]:
from fdcpa_classifier.eval import compute_metrics

all_results = {
    'o3-mini': openai_results,
    'Qwen Base': base_results,
    'Qwen QLoRA': ft_results,
}

print(f"{'Model':<20} {'Accuracy':>10} {'F1 (macro)':>12} {'Parse Rate':>12} {'Avg Latency':>12}")
print('-' * 68)
for name, results in all_results.items():
    m = compute_metrics(results)
    o = m['overall']
    print(f"{name:<20} {o['accuracy']:>10.3f} {o['f1_macro']:>12.3f} {o['parse_success_rate']:>12.1%} {o['avg_latency_ms']:>10.0f}ms")

## 6. Per-Rule F1 Chart

In [ ]:
from fdcpa_classifier.eval import save_results

serialized = save_results(all_results, output_path='../results/eval_results.json')

In [ ]:
from fdcpa_classifier.metrics import plot_per_rule_f1, plot_confusion_matrices, plot_cost_comparison

plot_per_rule_f1(serialized, output_path='../results/per_rule_f1.png')
plot_confusion_matrices(serialized, output_path='../results/confusion_matrices.png')
plot_cost_comparison(serialized, output_path='../results/cost_comparison.png')

## 7. Failure Analysis

In [ ]:
print("=" * 80)
print("FAILURE ANALYSIS: Fine-tuned model's worst failures")
print("=" * 80)

from fdcpa_classifier import load_jsonl
test_ex = load_jsonl('../data/test.jsonl')

failures = []
for i, (ex, pred) in enumerate(zip(test_ex, ft_results)):
    if pred['predicted'] != ex['verdict']:
        failures.append((i, ex, pred))

failures.sort(key=lambda x: len(x[1]['transcript_chunk']), reverse=True)

for idx, (i, ex, pred) in enumerate(failures[:5]):
    print(f"\n{'─'*80}")
    print(f"Failure #{idx+1} — Example {i}")
    print(f"Rule: {ex['rule_id']} | Actual: {ex['verdict']} | Predicted: {pred['predicted']}")
    print(f"\nTranscript:\n{ex['transcript_chunk'][:500]}...")
    print(f"\nModel raw output: {pred['raw_output'][:300]}")
    print(f"\nHypothesis: ")

## 8. Summary

In [ ]:
print("\n" + "=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print("""
1. [Fill in after running eval] Where fine-tuning helps most
2. [Fill in after running eval] Where fine-tuning doesn't help
3. [Fill in after running eval] Cost/latency tradeoff
4. [Fill in after running eval] Composability with Scrutiny
""")